## Notebook 03 — Credit Decisioning Engine

This notebook applies the PD estimates generated in Notebook 02 to construct
a simple but realistic credit decisioning framework.

Using predicted default risk, the notebook:
- Applies approve / decline rules based on risk appetite
- Assigns risk-based APR pricing tiers
- Recommends approved loan amounts (limits)
- Evaluates portfolio-level outcomes such as acceptance rate and bad rate

The objective is to demonstrate how PD models are used in practice to drive
credit policy, rather than to further optimise model performance.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../outputs/test_with_pd.parquet")

print(df.shape)
df.head()


(258933, 4)


,application_id,issue_d,target_bad,pd
0,892038,2016-11-01,0,0.332169
1,891225,2016-11-01,0,0.431273
2,891941,2016-11-01,1,0.333189
3,891974,2016-11-01,0,0.662782
4,892041,2016-11-01,0,0.691759


In [2]:
df["risk_band"] = pd.qcut(
    df["pd"],
    q=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["A", "B", "C", "D", "E"]
)

df["risk_band"].value_counts().sort_index()


risk_band
A    51787
B    51786
C    51787
D    51786
E    51787
Name: count, dtype: int64

Risk bands created as policy works on bands, not raw PDs.Risk bands were defined using quantile-based bucketing to ensure stable volume per band .

In [3]:
PD_CUTOFF = 0.35  # risk appetite threshold

df["decision"] = np.where(df["pd"] > PD_CUTOFF, "DECLINE", "APPROVE")

df["decision"].value_counts(normalize=True)


decision
DECLINE    0.793839
APPROVE    0.206161
Name: proportion, dtype: float64

In [4]:
apr_map = {
    "A": 8.9,
    "B": 12.9,
    "C": 16.9,
    "D": 21.9,
    "E": 29.9
}

df["apr"] = df["risk_band"].map(apr_map)


In [5]:
def assign_amount(row):
    if row["decision"] == "DECLINE":
        return 0
    if row["risk_band"] == "A":
        return 20000
    if row["risk_band"] == "B":
        return 15000
    if row["risk_band"] == "C":
        return 8000
    if row["risk_band"] == "D":
        return 4000
    return 1000

df["approved_amount"] = df.apply(assign_amount, axis=1)


In [6]:
def reason_code(row):
    if row["decision"] == "DECLINE":
        return "High predicted default risk"
    if row["risk_band"] in ["D", "E"]:
        return "Elevated risk – higher pricing and lower limit"
    return "Approved within standard risk appetite"

df["reason_code"] = df.apply(reason_code, axis=1)


In [7]:
approved = df[df["decision"] == "APPROVE"]

accept_rate = len(approved) / len(df)
approved_bad_rate = approved["target_bad"].mean()

print("Acceptance rate:", round(accept_rate, 3))
print("Bad rate (approved loans):", round(approved_bad_rate, 3))


Acceptance rate: 0.206
Bad rate (approved loans): 0.137


In [8]:
approved = approved.copy()

approved["pd"] = pd.to_numeric(approved["pd"], errors="coerce")
approved["apr"] = pd.to_numeric(approved["apr"], errors="coerce")
approved["approved_amount"] = pd.to_numeric(approved["approved_amount"], errors="coerce")
approved["target_bad"] = pd.to_numeric(approved["target_bad"], errors="coerce")


In [9]:
band_perf = (
    approved
    .groupby("risk_band")
    .agg(
        volume=("application_id", "count"),
        bad_rate=("target_bad", "mean"),
        avg_pd=("pd", "mean"),
        avg_apr=("apr", "mean"),
        avg_amount=("approved_amount", "mean")
    )
)

band_perf


C:\Users\adity\AppData\Local\Temp\ipykernel_19836\874851745.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("risk_band")


,volume,bad_rate,avg_pd,avg_apr,avg_amount
risk_band,,,,,
A,51787,0.136115,0.293698,8.9,20000.0
B,1595,0.171787,0.348915,12.9,15000.0
C,0,NaN,NaN,NaN,NaN
D,0,NaN,NaN,NaN,NaN
E,0,NaN,NaN,NaN,NaN


In [10]:
decision_output = df[[
    "application_id",
    "pd",
    "risk_band",
    "decision",
    "apr",
    "approved_amount",
    "reason_code",
    "target_bad"
]]

decision_output.to_parquet("../outputs/decision_output.parquet", index=False)
print("Saved decision_output.parquet")


Saved decision_output.parquet


In [11]:
import pandas as pd

df_decision = pd.read_parquet("../outputs/decision_output.parquet")

df_decision.head(10)


,application_id,pd,risk_band,decision,apr,approved_amount,reason_code,target_bad
0,892038,0.332169,A,APPROVE,8.9,20000,Approved within standard risk appetite,0
1,891225,0.431273,C,DECLINE,16.9,0,High predicted default risk,0
2,891941,0.333189,A,APPROVE,8.9,20000,Approved within standard risk appetite,1
3,891974,0.662782,E,DECLINE,29.9,0,High predicted default risk,0
4,892041,0.691759,E,DECLINE,29.9,0,High predicted default risk,0
5,892066,0.621464,E,DECLINE,29.9,0,High predicted default risk,0
6,891379,0.409225,B,DECLINE,12.9,0,High predicted default risk,0
7,892065,0.363191,B,DECLINE,12.9,0,High predicted default risk,0
8,892063,0.451558,C,DECLINE,16.9,0,High predicted default risk,0
9,891929,0.321504,A,APPROVE,8.9,20000,Approved within standard risk appetite,0


This table shows the output of an end-to-end credit decisioning engine applied counterfactually to historical loan applications.

For each application, the engine:

Estimates Probability of Default (PD)

Assigns a risk band

Applies a credit policy to approve or decline

Determines pricing (APR) and approved loan amount

Compares the decision to the observed loan outcome (target_bad)

The goal is not to predict outcomes perfectly loan-by-loan, but to show how model outputs translate into portfolio-level decisions and risk control.

Take row 0 Application ID 892038 as an example:

At application time, the model estimated a 33% probability of default

This falls within the lender’s risk appetite (below the PD cutoff)

The loan was approved, priced at a low APR, and granted a high amount

The borrower ultimately did not default

This is a successful lending outcome under the policy.

Take ROW 1 Application ID 891225:
This represents a false positive:

The model predicted high risk

The policy declined the loan

In hindsight, the borrower would have performed

Take Row 2 Application ID  891941 :
This represents a false negative:

The loan appeared acceptable at application

It was approved

It later defaulted

This table illustrates how predicted default risk is translated into real lending decisions, pricing, and credit limits, and how those decisions perform when evaluated against observed loan outcomes at a portfolio level.
The decisioning output illustrates the trade-offs inherent in credit risk
management. Some declined applications would have performed well, while some
approved loans subsequently defaulted. These outcomes reflect intentional
policy choices driven by risk appetite rather than model error.

## Conclusion

This notebook demonstrates how a Probability of Default (PD) model can be
translated into a practical, end-to-end credit decisioning framework. Rather
than treating the PD model as an isolated predictive tool, it is used as an
input into lender-side policy decisions that govern approvals, pricing, and
credit limits.

Using quantile-based risk bands, predicted default risk is converted into
transparent pricing tiers and approved loan amounts. A simple PD threshold is
then applied to represent the lender’s risk appetite, producing approve/decline
decisions that are internally consistent and easy to interpret. The resulting
decisioning output mirrors real-world credit workflows, where models inform
policy rather than replacing it.

Evaluating decisions against observed loan outcomes highlights the inherent
trade-offs in credit risk management. Some declined applications would have
performed well, while some approved loans subsequently defaulted. These outcomes
reflect intentional policy choices and probabilistic uncertainty, rather than
model error, and emphasise the importance of portfolio-level evaluation over
loan-by-loan accuracy.

Overall, this exercise shows how predictive risk models, policy rules, and
business constraints interact to shape lending outcomes. The framework provides
a foundation that could be extended with expected loss calculations, profit
optimisation, or borrower take-up modelling, but already captures the core
principles of modern consumer credit decisioning.


In [20]:


df_eval = df.copy()

df_eval["Outcome"] = np.select(
    [
        (df_eval["decision"] == "APPROVE") & (df_eval["target_bad"] == 0),
        (df_eval["decision"] == "APPROVE") & (df_eval["target_bad"] == 1),
        (df_eval["decision"] == "DECLINE") & (df_eval["target_bad"] == 0),
        (df_eval["decision"] == "DECLINE") & (df_eval["target_bad"] == 1),
    ],
    [
        "Approved & Good",
        "Approved & Bad",
        "Declined but Good",
        "Declined & Bad",
    ],
    default="Other"   
)

outcome_table = pd.DataFrame({
    "Count": df_eval["Outcome"].value_counts(),
    "Percentage (%)": (
        df_eval["Outcome"].value_counts(normalize=True)
        .mul(100)
        .round(1)
    )
})

outcome_table





,Count,Percentage (%)
Outcome,,
Declined but Good,144392,55.8
Declined & Bad,61159,23.6
Approved & Good,46059,17.8
Approved & Bad,7323,2.8


### Policy Outcomes and Trade-offs

The table below summarises lending outcomes under the applied credit policy.
Approximately 20% of applications are approved, reflecting a conservative risk
appetite. The majority of declined applications would have performed well
("Declined but Good"), representing intentional false positives.

Only a small proportion of total applications (approximately 3%) are both
approved and subsequently default, indicating that losses within the approved
portfolio are tightly controlled. These results highlight the inherent trade-
off in credit decisioning between maximising approval rates and maintaining
portfolio credit quality.
